# 01 - Feature Engineering

Goal: run the real feature extraction pipeline (lexical WER via spaCy, phoneme mismatch via CMU dict, MFCC + pause ratio via librosa) on sample inputs and validate the outputs look sane.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, '..')
from src.nlp.feature_extractor import extract_features
from src.speech_to_text.text_cleaning import clean_transcript, tokenize

REAL_AUDIO_DIR = Path('../data/sample_audio/real_samples')
transcripts = json.loads((REAL_AUDIO_DIR / 'transcripts.json').read_text())
clip_id, transcript = next(iter(transcripts.items()))
prompt_words = tokenize(clean_transcript(transcript))
prompt_words

['many',
 'little',
 'wrinkles',
 'gathered',
 'between',
 'his',
 'eyes',
 'as',
 'he',
 'contemplated',
 'this',
 'and',
 'his',
 'brow',
 'moistened']

### Perfect reading attempt (spoken == prompt)

In [2]:
res_perfect = extract_features(
    prompt_words, prompt_words, duration_sec=3.0,
    audio_path=REAL_AUDIO_DIR / f'{clip_id}.wav',
)
res_perfect.features

/private/tmp/claude-501/-Users-krisharathod-Desktop/56941ab0-c436-452e-accc-b3b3744394d8/scratchpad/neuroaid-ai/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'wpm': 300.0,
 'word_error_rate': 0.0,
 'phoneme_mismatch_proxy': 0.0,
 'spoken_word_count': 15.0,
 'prompt_word_count': 15.0,
 'mfcc_0_mean': -302.2296142578125,
 'mfcc_1_mean': 115.11511993408203,
 'mfcc_2_mean': -2.7817535400390625,
 'mfcc_3_mean': -6.535029411315918,
 'mfcc_4_mean': -33.78394317626953,
 'mfcc_5_mean': -17.789772033691406,
 'mfcc_6_mean': -1.0355387926101685,
 'mfcc_7_mean': -22.743030548095703,
 'mfcc_8_mean': -7.779958724975586,
 'mfcc_9_mean': 4.877002239227295,
 'mfcc_10_mean': -3.716700792312622,
 'mfcc_11_mean': -7.317237377166748,
 'mfcc_12_mean': -9.291494369506836,
 'mfcc_0_std': 94.26113891601562,
 'mfcc_1_std': 34.27892303466797,
 'mfcc_2_std': 42.20999526977539,
 'mfcc_3_std': 28.732471466064453,
 'mfcc_4_std': 23.91645050048828,
 'mfcc_5_std': 17.697261810302734,
 'mfcc_6_std': 11.71692943572998,
 'mfcc_7_std': 13.39337158203125,
 'mfcc_8_std': 10.091179847717285,
 'mfcc_9_std': 7.423564434051514,
 'mfcc_10_std': 7.274589538574219,
 'mfcc_11_std': 10.0

### Simulated struggling reading attempt (words dropped/substituted)

This is the same kind of perturbation `scripts/generate_training_data.py` uses to synthesize label diversity from a small set of real clips.

In [3]:
struggling_words = prompt_words[: max(1, len(prompt_words) - 3)] + ['um', 'uh']
res_struggling = extract_features(
    prompt_words, struggling_words, duration_sec=6.0,
    audio_path=REAL_AUDIO_DIR / f'{clip_id}.wav',
)
res_struggling.features

{'wpm': 140.0,
 'word_error_rate': 0.2,
 'phoneme_mismatch_proxy': 0.13636363636363635,
 'spoken_word_count': 14.0,
 'prompt_word_count': 15.0,
 'mfcc_0_mean': -302.2296142578125,
 'mfcc_1_mean': 115.11511993408203,
 'mfcc_2_mean': -2.7817535400390625,
 'mfcc_3_mean': -6.535029411315918,
 'mfcc_4_mean': -33.78394317626953,
 'mfcc_5_mean': -17.789772033691406,
 'mfcc_6_mean': -1.0355387926101685,
 'mfcc_7_mean': -22.743030548095703,
 'mfcc_8_mean': -7.779958724975586,
 'mfcc_9_mean': 4.877002239227295,
 'mfcc_10_mean': -3.716700792312622,
 'mfcc_11_mean': -7.317237377166748,
 'mfcc_12_mean': -9.291494369506836,
 'mfcc_0_std': 94.26113891601562,
 'mfcc_1_std': 34.27892303466797,
 'mfcc_2_std': 42.20999526977539,
 'mfcc_3_std': 28.732471466064453,
 'mfcc_4_std': 23.91645050048828,
 'mfcc_5_std': 17.697261810302734,
 'mfcc_6_std': 11.71692943572998,
 'mfcc_7_std': 13.39337158203125,
 'mfcc_8_std': 10.091179847717285,
 'mfcc_9_std': 7.423564434051514,
 'mfcc_10_std': 7.274589538574219,
 'mf

Note the lower `wpm` (slower, from the longer duration) and higher `word_error_rate` for the struggling attempt vs. the perfect one -- these are the signals `src/model/predict.py` turns into a plain-language risk band.